In [ ]:
!pip install requests

In [ ]:
import requests
from datetime import datetime, timedelta

# Constants
GITHUB_API_URL = "https://api.github.com/search/repositories"
QUERY = "language:Solidity"
MIN_STARS = 500  # Define what you mean by "high stars"
MONTHS = 6  # Define the time frame in months
DATE_FORMAT = "%Y-%m-%dT%H:%M:%SZ"
EXCLUDED_KEYWORDS = ["hack", "tutorial", "example", "course", "learning"]

# Calculate the date 6 months ago
six_months_ago = (datetime.utcnow() - timedelta(days=30 * MONTHS)).strftime(DATE_FORMAT)

# Query GitHub API
params = {
    "q": f"{QUERY} stars:>{MIN_STARS} pushed:>={six_months_ago}",
    "sort": "stars",
    "order": "desc",
    "per_page": 100  # Adjust the number of results per page as needed
}
headers = {
    "Accept": "application/vnd.github.v3+json"
}

response = requests.get(GITHUB_API_URL, params=params, headers=headers)
data = response.json()

# Filter out non-protocol repositories
filtered_repos = [
    repo for repo in data.get("items", [])
    if not any(keyword in (repo.get("name", "") + " " + (repo.get("description") or "")).lower() for keyword in EXCLUDED_KEYWORDS)
]

# Display the filtered repositories
for repo in filtered_repos:
    print(f"Name: {repo['name']}")
    print(f"Full Name: {repo['full_name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}")
    print(f"Updated At: {repo['updated_at']}")
    print(f"URL: {repo['html_url']}")
    print("\n")


In [ ]:
len(filtered_repos)

In [ ]:
import requests
from datetime import datetime, timedelta

# Constants
GITHUB_API_URL = "https://api.github.com/search/repositories"
QUERY = "language:Solidity"
MIN_STARS = 500  # Define what you mean by "high stars"
MONTHS = 6  # Define the time frame in months
DATE_FORMAT = "%Y-%m-%dT%H:%M:%SZ"

# Keywords for popular protocols and libraries
POPULAR_PROTOCOLS = [
    "aave", "compound", "uniswap", "balancer", "curve", "chainlink",
    "synthetix", "yearn", "gnosis", "openzeppelin", "0x", "aragon",
    "set protocol", "tornado cash", "ens", "uma", "opium", "lens"
]

# Calculate the date 6 months ago
six_months_ago = (datetime.utcnow() - timedelta(days=30 * MONTHS)).strftime(DATE_FORMAT)

# Query GitHub API
params = {
    "q": f"{QUERY} stars:>{MIN_STARS} pushed:>={six_months_ago}",
    "sort": "stars",
    "order": "desc",
    "per_page": 100  # Adjust the number of results per page as needed
}
headers = {
    "Accept": "application/vnd.github.v3+json"
}

response = requests.get(GITHUB_API_URL, params=params, headers=headers)
data = response.json()

# Filter repositories to include only known popular protocols and libraries
filtered_repos = [
    repo for repo in data.get("items", [])
    if any(keyword in (repo.get("name", "") + " " + (repo.get("description") or "") + " " + " ".join(repo.get("topics", []))).lower() for keyword in POPULAR_PROTOCOLS)
]

# Display the filtered repositories
for repo in filtered_repos:
    print(f"Name: {repo['name']}")
    print(f"Full Name: {repo['full_name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}")
    print(f"Updated At: {repo['updated_at']}")
    print(f"URL: {repo['html_url']}")
    print("\n")


In [ ]:
import requests
from datetime import datetime, timedelta

# List of DeFi projects extracted from the Alchemy page (hypothetical example)
defi_projects = [
    {"name": "Aave", "github": "https://github.com/aave/aave-protocol"},
    {"name": "Uniswap", "github": "https://github.com/Uniswap/uniswap-v2-core"},
    {"name": "Compound", "github": "https://github.com/compound-finance/compound-protocol"},
    # Add other projects here
]

# Function to check if a project is actively maintained
def is_actively_maintained(repo_url):
    repo_name = repo_url.rstrip('/').split('/')[-2:]
    repo_name = '/'.join(repo_name)
    
    api_url = f"https://api.github.com/repos/{repo_name}/commits"
    headers = {"Accept": "application/vnd.github.v3+json"}
    
    response = requests.get(api_url, headers=headers)
    if response.status_code != 200:
        return False
    
    commits = response.json()
    if not commits:
        return False
    
    latest_commit_date = commits[0]["commit"]["committer"]["date"]
    latest_commit_date = datetime.strptime(latest_commit_date, "%Y-%m-%dT%H:%M:%SZ")
    six_months_ago = datetime.utcnow() - timedelta(days=6*30)
    
    return latest_commit_date > six_months_ago

# Check each project and print the actively maintained ones
active_projects = []
for project in defi_projects:
    if is_actively_maintained(project["github"]):
        active_projects.append(project)

# Display the actively maintained projects
for project in active_projects:
    print(f"Name: {project['name']}")
    print(f"GitHub: {project['github']}")
    print("\n")


In [ ]:
import os
import re
import subprocess
import pandas as pd
from pathlib import Path

# List of repositories to analyze
repositories = [
    {"url": "https://github.com/lidofinance/lido-dao", "path": "lido-dao"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/Layr-Labs/eigenlayer-contracts", "path": "eigenlayer-contracts"},
    {"url": "https://github.com/makerdao/dss", "path": "dai-stablecoin-system"},
    {"url": "https://github.com/rocket-pool/rocketpool", "path": "rocketpool"},
    {"url": "https://github.com/Uniswap/v3-core", "path": "uniswap-v3-core"},
    {"url": "https://github.com/Instadapp/dsa-resolvers", "path": "instadapp-dsa-resolvers"},
    {"url": "https://github.com/Instadapp/dsa-connectors", "path": "instadapp-dsa-connectors"},
    {"url": "https://github.com/compound-finance/compound-protocol", "path": "compound-protocol"},
    {"url": "https://github.com/aave/protocol-v2", "path": "aave-protocol-v2"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/VenusProtocol/venus-protocol", "path": "venus-protocol"},
    {"url": "https://github.com/pancakeswap/pancake-smart-contracts", "path": "pancake-smart-contracts"},
    {"url": "https://github.com/pancakeswap/pancake-v4-core", "path": "pancake-v4-core"},
    {"url": "https://github.com/FraxFinance/frax-solidity", "path": "frax-solidity"},
    {"url": "https://github.com/pancakeswap/pancake-v3-contracts", "path": "pancake-v3-contracts"},
    {"url": "https://github.com/aerodrome-finance/contracts", "path": "aerodrome-finance-contracts"}
]

# Function to clone a repository
def clone_repository(repo_url, repo_path):
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", repo_url, repo_path])

# Function to extract require and assert statements from a file
def extract_statements(file_path):
    with open(file_path, 'r') as file:
        content = file.read()

    # Remove comments
    content = re.sub(r'//.*|/\*[\s\S]*?\*/', '', content)

    # Regex to find require and assert statements with optional message
    pattern = re.compile(r'(require|assert)\s*\((.*?)\)\s*;')
    statements = pattern.findall(content)

    # Regex to find pragma solidity
    pragma_pattern = re.compile(r'pragma solidity ([^;]+);')
    pragma = pragma_pattern.search(content)
    pragma_version = pragma.group(1) if pragma else ""

    return statements, pragma_version, content

# Function to analyze the context of the require/assert statement
def analyze_context(content, statement):
    # Determine the code block type and placement
    function_pattern = re.compile(r'(function\s+\w+\s*\(.*?\))')
    modifier_pattern = re.compile(r'(modifier\s+\w+\s*\(.*?\))')
    contract_pattern = re.compile(r'(contract\s+\w+)')

    code_block = ""
    placement = "middle"
    function_type = ""
    function_visibility = ""
    contract_id = ""
    block_id = ""

    # Find all contracts
    contracts = list(contract_pattern.finditer(content))
    for contract in contracts:
        start = contract.start()
        end = content.find('}', start) + 1
        if statement in content[start:end]:
            contract_id = contract.group().split()[1]
            break

    # Find all functions and modifiers within the identified contract
    functions_modifiers = list(function_pattern.finditer(content)) + list(modifier_pattern.finditer(content))
    for fm in functions_modifiers:
        start = fm.start()
        end = content.find('}', start) + 1
        if statement in content[start:end]:
            block_content = content[start:end]
            if fm.group().startswith('function'):
                code_block = "function"
                block_id = fm.group().split()[1]
            else:
                code_block = "modifier"
                block_id = fm.group().split()[1]

            statement_index = block_content.find(statement)

            # Determine placement
            if block_content[:statement_index].strip() == '':
                placement = "beginning"
            elif block_content[statement_index + len(statement):].strip() == '':
                placement = "end"
            else:
                placement = "middle"
            break

    return code_block, placement, function_type, function_visibility, contract_id, block_id

# Traverse the directory and extract statements
data = []
protocols_dir = Path("protocols")
protocols_dir.mkdir(exist_ok=True)
for repo in repositories:
    repo_path = protocols_dir / repo["path"]
    clone_repository(repo["url"], str(repo_path))
    for root, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".sol"):
                file_path = os.path.join(root, file)
                statements, pragma_version, content = extract_statements(file_path)
                for statement, predicate in statements:
                    message = ""
                    if "," in predicate:
                        predicate, message = predicate.rsplit(",", 1)
                    code_block, placement, function_type, function_visibility, contract_id, block_id = analyze_context(content, statement)
                    data.append({
                        "protocol": repo["path"],
                        "predicate": predicate.strip(),
                        "message": message.strip(),
                        "code_block": code_block,
                        "placement": placement,
                        "pragma_version": pragma_version,
                        "function_type": function_type,
                        "function_visibility": function_visibility,
                        "contract_id": contract_id,
                        "block_id": block_id
                    })

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data)
output_dir = Path("datasets")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "require_assert_statements.csv"
df.to_csv(output_path, index=False)

print(f"Data extracted and saved to {output_path}")
print(df.head(10))


## The better version to build require/assert statements

In [ ]:
import os
import re
import subprocess
import pandas as pd
from pathlib import Path

# List of repositories to analyze
repositories = [
    {"url": "https://github.com/lidofinance/lido-dao", "path": "lido-dao"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/Layr-Labs/eigenlayer-contracts", "path": "eigenlayer-contracts"},
    {"url": "https://github.com/makerdao/dss", "path": "dai-stablecoin-system"},
    {"url": "https://github.com/rocket-pool/rocketpool", "path": "rocketpool"},
    {"url": "https://github.com/Uniswap/v3-core", "path": "uniswap-v3-core"},
    {"url": "https://github.com/Instadapp/dsa-resolvers", "path": "dsa-resolvers"},
    {"url": "https://github.com/Instadapp/dsa-connectors", "path": "dsa-connectors"},
    {"url": "https://github.com/compound-finance/compound-protocol", "path": "compound-protocol"},
    {"url": "https://github.com/aave/protocol-v2", "path": "protocol-v2"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/VenusProtocol/venus-protocol", "path": "venus-protocol"},
    {"url": "https://github.com/pancakeswap/pancake-smart-contracts", "path": "pancake-smart-contracts"},
    {"url": "https://github.com/pancakeswap/pancake-v4-core", "path": "pancake-v4-core"},
    {"url": "https://github.com/FraxFinance/frax-solidity", "path": "frax-solidity"},
    {"url": "https://github.com/pancakeswap/pancake-v3-contracts", "path": "pancake-v3-contracts"},
    {"url": "https://github.com/aerodrome-finance/contracts", "path": "contracts"}
]

# Function to clone a repository
def clone_repository(repo_url, repo_path):
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", repo_url, repo_path])

# Function to remove comments from Solidity code
def remove_comments(content):
    content = re.sub(r'//.*', '', content)
    content = re.sub(r'/\*[\s\S]*?\*/', '', content)
    return content

# Function to extract require and assert statements from a file
def extract_statements(file_path):
    with open(file_path, 'r') as file:
        content = file.read()

    # Remove comments
    content = remove_comments(content)

    # Regex to find require and assert statements with optional message
    pattern = re.compile(r'(require|assert)\s*\((.*?)\)\s*;')
    statements = pattern.findall(content)

    # Regex to find pragma solidity
    pragma_pattern = re.compile(r'pragma solidity ([^;]+);')
    pragma = pragma_pattern.search(content)
    pragma_version = pragma.group(1) if pragma else ""

    return statements, pragma_version, content

# Function to analyze the context of the require/assert statement
def analyze_context(content, statement):
    # Determine the code block type and placement
    function_pattern = re.compile(r'function\s+\w+\s*\(.*?\)\s*(public|private|internal|external)?\s*(view|pure)?')
    modifier_pattern = re.compile(r'modifier\s+\w+\s*\(.*?\)')
    contract_pattern = re.compile(r'contract\s+\w+')

    code_block = ""
    placement = "middle"
    function_type = ""
    function_visibility = ""
    contract_id = ""
    block_id = ""

    # Find all contracts
    contracts = list(contract_pattern.finditer(content))
    for contract in contracts:
        start = contract.start()
        end = content.find('}', start) + 1
        if statement in content[start:end]:
            contract_id = contract.group().split()[1]
            break

    # Find all functions and modifiers within the identified contract
    functions_modifiers = list(function_pattern.finditer(content)) + list(modifier_pattern.finditer(content))
    for fm in functions_modifiers:
        start = fm.start()
        end = content.find('}', start) + 1
        if statement in content[start:end]:
            block_content = content[start:end]
            if fm.group().startswith('function'):
                code_block = "function"
                block_id = fm.group().split()[1]
            else:
                code_block = "modifier"
                block_id = fm.group().split()[1]

            statement_index = block_content.find(statement)

            # Determine placement
            block_lines = block_content.splitlines()
            statement_line = next(i for i, line in enumerate(block_lines) if statement in line)
            non_empty_lines = [line for line in block_lines if line.strip()]
            if statement_line == 0:
                placement = "beginning"
            elif statement_line == len(non_empty_lines) - 1:
                placement = "end"
            else:
                placement = "middle"
            break

    return code_block, placement, function_type, function_visibility, contract_id, block_id

# Traverse the directory and extract statements
data = []
protocols_dir = Path("protocols")
protocols_dir.mkdir(exist_ok=True)
for repo in repositories:
    repo_path = protocols_dir / repo["path"]
    clone_repository(repo["url"], str(repo_path))
    for root, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".sol"):
                file_path = os.path.join(root, file)
                statements, pragma_version, content = extract_statements(file_path)
                for statement_type, predicate in statements:
                    message = ""
                    if "," in predicate:
                        predicate, message = predicate.rsplit(",", 1)
                    code_block, placement, function_type, function_visibility, contract_id, block_id = analyze_context(content, statement_type + '(' + predicate + ');')
                    data.append({
                        "protocol": repo["path"],
                        "statement-type": statement_type,
                        "predicate": predicate.strip(),
                        "message": message.strip(),
                        "code_block": code_block,
                        "placement": placement,
                        "pragma_version": pragma_version,
                        "function_type": function_type,
                        "function_visibility": function_visibility,
                        "contract_id": contract_id,
                        "block_id": block_id,
                        "path": file_path
                    })

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data)
output_dir = Path("datasets")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "require_assert_statements.csv"
df.to_csv(output_path, index=False)

print(f"Data extracted and saved to {output_path}")
print(df.head(10))


In [88]:


import re

def replace_comments_with_newlines(match):
    # Determine if it's a multiline comment/docstring
    text = match.group(0)
    if text.startswith('/*') or text.startswith('/**'):
        # Count the number of newlines and replace with equivalent number of newlines
        return '\n' * text.count('\n')
    elif text.startswith('//'):
        # For single-line comments, replace with a space (or nothing if you prefer)
        return ' ' if '\n' in text else ''
    else:
        # For strings, return them unchanged
        return text

def remove_NatSpec_and_comments(src):
    # Updated pattern to match docstrings (/** ... */), multiline comments (/* ... */), single-line comments (// ...),
    # and strings (both "..." and '...')
    pattern = r'\/\*\*[\s\S]*?\*\/|\/\*[\s\S]*?\*\/|\/\/[^\n]*|$|".*?"|\'.*?\''
    src = re.sub(pattern, replace_comments_with_newlines, src, flags=re.MULTILINE)
    return src


class StackEmptyError(Exception):
    """Exception raised when an operation is attempted on an empty stack."""
    def __init__(self, message="Stack is empty!"):
        self.message = message
        super().__init__(self.message)

class Stack:
    def __init__(self):
        self.items = []

    def is_empty(self):
        return self.items == []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        if not self.is_empty():
            return self.items.pop()
        else:
            raise StackEmptyError("Attempt to pop from empty stack")

    def peek(self):
        if not self.is_empty():
            return self.items[-1]
        else:
            raise StackEmptyError("Cannot peek, stack is empty!")

    def size(self):
        return len(self.items)


class Explorer:
    DEBUG = False
    BLOCKS = [
        "function",
        "contract",
        "library",
        "abstract contract",
        "interface",
        "while",
        "for",
        "assembly",
        "unchecked"
    ]

    @staticmethod
    def dprint(arg):
        if Explorer.DEBUG:
            print(arg)
    


    @staticmethod
    def get_node_info(statement: str) -> dict:
        #print(f'finding node info for statement: {statement}')
        enclosing_node = {
            'type': None,
            'name': None
        }
        if "function" in statement:

            if 'function' in statement.strip().split(' '):
                print(f'found a function!     -----> {statement}')
                #print('just found a function in the statement!')
                keyword_index = statement.strip().split(' ').index('function')
                #print(f'keyword_index: {keyword_index}')
                fn_name = statement.strip().split(' ')[keyword_index + 1].split('(')[0]
                #print(f'fn_name: {fn_name}')
                enclosing_node.update({
                    'type': 'function',
                    'name': fn_name
                })
            elif 'function()' in statement.strip().split(' '):
                print(f'found a fallback!     -----> {statement}')
                keyword_index = statement.strip().split(' ').index('function()')
                fn_name = 'fallback'
                enclosing_node.update({
                    'type': 'function',
                    'name': fn_name
                })
        elif "receive()" in statement:
            print(f'found a receive!     -----> {statement}')
            enclosing_node.update({
                'type': 'function',
                'name': 'receive'
            })
        elif "fallback()" in statement:
            print(f'found a fallback!     -----> {statement}')
            enclosing_node.update({
                'type': 'function',
                'name': 'fallback'
            })
        elif "constructor" in statement:
            print(f'found a constructor!     -----> {statement}')
            enclosing_node.update({
                'type': 'constructor',
                'name': 'constructor'
            })
        elif "contract" in statement:
            print(f'found a contract!     -----> {statement}')
            if 'contract' in statement.strip().split(' '):
                keyword_index = statement.strip().split(' ').index('contract')
                contract_name = statement.strip().split(' ')[keyword_index + 1]
                enclosing_node.update({
                    'type': 'contract',
                    'name': contract_name
                })
        elif "interface" in statement:
            print(f'found an interface!     -----> {statement}')
            if 'interface' in statement.strip().split(' '):
                keyword_index = statement.strip().split(' ').index('interface')
                int_name = statement.strip().split(' ')[keyword_index + 1]
                enclosing_node.update({
                    'type': 'interface',
                    'name': int_name
                })
        elif "library" in statement:
            print(f'found a library!     -----> {statement}')
            if 'library' in statement.strip().split(' '):
                keyword_index = statement.strip().split(' ').index('library')
                lib_name = statement.strip().split(' ')[keyword_index + 1]
                enclosing_node.update({
                    'type': 'library',
                    'name': lib_name
                })
        elif "abstract contract" in statement:
            print(f'found an abstract contract!     -----> {statement}')
            if 'abstract' in statement.strip().split(' '):
                keyword_index = statement.strip().split(' ').index('abstract')
                contract_name = statement.strip().split(' ')[keyword_index + 2]
                enclosing_node.update({
                    'type': 'abstract contract',
                    'name': contract_name
                })
        elif "modifier" in statement:
            print(f'found a modifier!     -----> {statement}')
            if 'modifier' in statement.strip().split(' '):
                keyword_index = statement.strip().split(' ').index('modifier')
                modifier_name = statement.strip().split
                enclosing_node.update({
                    'type': 'modifier',
                    'name': modifier_name
                })

        return enclosing_node

    @staticmethod
    def get_enclosing_nodes(src, start, end) -> dict:
        Explorer.dprint(f"\nstart: {start}, end: {end}\n")
        
        # if end is None:
        #     end = start + 1
        
        if start > end:
            raise ValueError("Start line cannot be greater than the end line.")
        
        
        function_keywords = [
            "constructor",
            "function"
        ]

        modifier_keywords = [
            "modifier"
        ]

        contract_keywords = [
            "contract",
            "library",
            "abstract contract",
            "interface"
        ]

        stack = Stack()

        

        src_lines = src.split('\n')
        len_src_lines = len(src_lines)
        
        # there is a difference between difinition start and block start of the unit; we point to block start with start variable
        definition_start = start


        if any(element in src_lines[start] for element in ['contract', 'function', 'abstract contract', 'interface', 'library']):
            while "{" not in src_lines[start]:
                start += 1                

        end = start + 1

        up = start + 1
        down = end - 1

        to_find = ['function', 'contract', 'library', 'abstract contract', 'interface']
        enclosing_nodes = {
            'function': None,
            'contract': None
        }
        
        direction = 'down'
        c_down = 0
        c_up = 0

        while down < len_src_lines - 1 or up > 0:
            if direction == 'down':
                down += 1
                Explorer.dprint(f'-    -     -    -     -    -     -    -     -   c_down: {c_down}, down: {down}')
                if down > len_src_lines - 1:
                    Explorer.dprint(f"Reaching the end of the file, going up (next up is {up-1})... ---- stack items: {stack.items}")
                    direction = 'up'
                    continue
                if "{" in src_lines[down] and "}" in src_lines[down]:
                    pass
                elif "}" in src_lines[down] and c_down == 0:                    
                    stack.push({
                        'token': "}",
                        'line': down
                    })
                    Explorer.dprint(f"Just saw a }} going down at line {down}, going up (next up is {up-1}) ---- stack items: {stack.items}")
                    direction = 'up'
                elif "{" in src_lines[down]:
                    Explorer.dprint(f"Saw a {{ at line {down} (going down), and skipping it... ---- stack items: {stack.items}")
                    c_down += 1
                elif "}" in src_lines[down] and c_down > 0:
                    Explorer.dprint(f"Saw a }} at line {down} (going down), and skipping it... ---- stack items: {stack.items}")
                    c_down -= 1
            if direction == 'up':
                
                #if up <= 1245:
                    #print(f'+++ up: {up} line: {src_lines[up]}')
                    

                up -= 1
                if up <= 0:
                    Explorer.dprint(f"Reaching the begining of the file going up, going down (next down is {down+1}) ---- stack items: {stack.items}")
                    direction = 'down'
                    continue

                if "{" in src_lines[up] and "}" in src_lines[up]:
                    pass    
                elif "{" in src_lines[up] and c_up == 0:
                    Explorer.dprint('we found one!!!')
                    # trying to see if the function, contract, etc. identifier exists in the same line as { is
                    any_blocks = any(element in src_lines[up] for element in Explorer.BLOCKS)
                    while not any_blocks:
                        # go up so you find anything
                        direction = 'up'
                        up -= 1
                        any_blocks = any(element in src_lines[up] for element in Explorer.BLOCKS)

                    stack.push({
                        'token': "{",
                        'line': up
                    })
                    top = stack.pop()
                    found_statement = src_lines[top['line']]                    
                    start_of_found_statement = top['line']

                    top = stack.pop()
                    end_of_found_statement = top['line']
                    
                    desirble = any(element in found_statement for element in function_keywords+contract_keywords+modifier_keywords)
                    if desirble:
                        #print(f'found_statement: {found_statement}')
                        statement_info = Explorer.get_node_info(found_statement)
                        #print(f'the current to_find list is: {to_find}, and we are trying to remove type of `{statement_info}`')
                        if statement_info['type'] is None:
                            return None
                        to_find.remove(statement_info['type']) 
                        enclosing_nodes[statement_info['type']] = {
                            'start_line': start_of_found_statement,
                            'end_line': end_of_found_statement,
                            'name': statement_info['name']
                        }

                        if len(to_find) > 0:
                            Explorer.dprint(f"Just identified an enclosing node (at line {up}) while moving up, going down now (next down is: {down+1})")

                            direction = 'down'
                            continue
                    elif len(to_find) > 0:
                        Explorer.dprint(f"Just identified an undesirable enclosing node (at line {up}) while moving up, going down now (next down is: {down+1}) ---- stack items: {stack.items}")                            
                        direction = 'down'
                        continue
                    else:
                        break
                elif "}" in src_lines[up]:
                    Explorer.dprint(f"Saw a }} going up (at line {up}); now skipping it... ---- stack items: {stack.items}")                    
                    c_up += 1
                    Explorer.dprint(f"{'^   '*10} c_up: {c_up}")
                elif "{" in src_lines[up] and c_up > 0:
                    Explorer.dprint(f"{'^   '*10}Saw a {{ going up (at line {up}); skpping it... ---- stack items: {stack.items}")
                    c_up -= 1
                    Explorer.dprint(f"{'^   '*10} c_up: {c_up}")
        
        return enclosing_nodes

    @staticmethod
    def detect_solidity_definition_and_name(src: str) -> dict:
        
        patterns = {
            'library': r'\blibrary\b\s+(\w+)\s*{',
            'abstract contract': r'\babstract\s+contract\b\s+(\w+)\s*{',
            'contract': r'\bcontract\b\s+(\w+)\s*{',
            'interface': r'\binterface\b\s+(\w+)\s*{',
            'function': r'\bfunction\b\s+(\w+)\s*',
            'modifier': r'\bmodifier\b\s+(\w+)\s*'
        }

        matches = []
        for definition, pattern in patterns.items():
            for match in re.finditer(pattern, src):
                name = match.group(1) if definition != 'constructor' else None
                matches.append((match.start(), definition, name))

        matches.sort(key=lambda x: x[0])

        if matches:
            match = matches[0]
            return {
                'type': match[1], 
                'name': match[2]
            }
        else:
            return {
                'type': None, 
                'name': None
            }
    

    @staticmethod
    def extract_code_context_with_line_numbers(full_code, start_line, end_line=None, context_span=20):
        # If end_line is None, set it to start_line
        if end_line is None:
            end_line = start_line
        
        # Ensure line numbers are indexed from 0 for internal processing
        start_line -= 1
        end_line -= 1
        
        # Split the full code into lines
        lines = full_code.split('\n')
        
        # Calculate start and end indices, making sure they are within the bounds of the source code lines
        start_index = max(0, start_line - context_span)
        end_index = min(len(lines), end_line + context_span + 1) # +1 to include the end line itself
        
        # Join the relevant lines back into a single string
        context = '\n'.join(lines[start_index:end_index])
        
        return context

In [89]:
import os
import re
import subprocess
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# List of repositories to analyze
repositories = [
    {"url": "https://github.com/lidofinance/lido-dao", "path": "lido-dao"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/Layr-Labs/eigenlayer-contracts", "path": "eigenlayer-contracts"},
    {"url": "https://github.com/makerdao/dss", "path": "dai-stablecoin-system"},
    {"url": "https://github.com/rocket-pool/rocketpool", "path": "rocketpool"},
    {"url": "https://github.com/Uniswap/v3-core", "path": "uniswap-v3-core"},
    {"url": "https://github.com/Instadapp/dsa-resolvers", "path": "dsa-resolvers"},
    {"url": "https://github.com/Instadapp/dsa-connectors", "path": "dsa-connectors"},
    {"url": "https://github.com/compound-finance/compound-protocol", "path": "compound-protocol"},
    {"url": "https://github.com/aave/protocol-v2", "path": "protocol-v2"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/VenusProtocol/venus-protocol", "path": "venus-protocol"},
    {"url": "https://github.com/pancakeswap/pancake-smart-contracts", "path": "pancake-smart-contracts"},
    {"url": "https://github.com/pancakeswap/pancake-v4-core", "path": "pancake-v4-core"},
    {"url": "https://github.com/FraxFinance/frax-solidity", "path": "frax-solidity"},
    {"url": "https://github.com/pancakeswap/pancake-v3-contracts", "path": "pancake-v3-contracts"},
    {"url": "https://github.com/aerodrome-finance/contracts", "path": "contracts"}
]

# Function to clone a repository
def clone_repository(repo_url, repo_path):
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", repo_url, repo_path])

# Function to remove comments from Solidity code
def remove_comments(content):
    return remove_NatSpec_and_comments(content)

# Function to extract require and assert statements from a file
def extract_statements(file_path):
    with open(file_path, 'r') as file:
        content = file.read()

    # Remove comments
    content = remove_comments(content)

    # Regex to find require and assert statements with optional message
    pattern = re.compile(r'(require|assert)\s*\((.*?)\)\s*;')
    statements = pattern.findall(content)

    # Regex to find pragma solidity
    pragma_pattern = re.compile(r'pragma solidity ([^;]+);')
    pragma = pragma_pattern.search(content)
    pragma_version = pragma.group(1) if pragma else ""

    return statements, pragma_version, content

# Function to analyze the context of the require/assert statement
def analyze_context(content, statement_type, predicate):
    # Combine the statement type and predicate for full statement
    statement = f"{statement_type}({predicate});"

    # Determine the code block type and placement
    lines = content.split('\n')

    # Locate the statement line
    statement_line = None
    for i, line in enumerate(lines):
        if statement_type in line and predicate in line:
            statement_line = i
            break

    if statement_line is None:
        return "", "", "", ""
    #print(f'getting enclosing nodes for statement: {statement}, at line: {statement_line}')

    try:
        enclosing_nodes = Explorer.get_enclosing_nodes(content, statement_line, statement_line)
    except Exception as e:
        #print(f"Error: {e}")
        return None
    #print(f"Enclosing nodes for statement '{statement}' at line {statement_line}: {enclosing_nodes}")

    # Safeguard the extraction of 'type' and 'name' fields
    function_node = enclosing_nodes.get('function') if enclosing_nodes else None
    contract_node = enclosing_nodes.get('contract') if enclosing_nodes else None

    #print(f'enclosing nodes for statement: {statement} are: {enclosing_nodes}')
    code_block = function_node.get('type', "") if function_node else ""
    contract_id = contract_node.get('name', "") if contract_node else ""
    block_id = function_node.get('name', "") if function_node else ""
    
    #print(f"code_block: {code_block}, contract_id: {contract_id}, block_id: {block_id}")

    placement = "middle"
    if function_node:
        if function_node.get('start_line') <= statement_line <= function_node.get('start_line') + 2:
            placement = "beginning"
        elif function_node.get('end_line')-2 <= statement_line <= function_node.get('end_line'):
            placement = "end"

    return code_block, placement, contract_id, block_id

# Traverse the directory and extract statements
data = []
protocols_dir = Path("protocols")
protocols_dir.mkdir(exist_ok=True)

# Adding tqdm for progress indication
for repo in tqdm(repositories, desc="Cloning repositories"):
    repo_path = protocols_dir / repo["path"]
    clone_repository(repo["url"], str(repo_path))

for repo in tqdm(repositories, desc="Processing repositories"):
    repo_path = protocols_dir / repo["path"]
    repo_files = [os.path.join(root, file) for root, _, files in os.walk(repo_path) for file in files if file.endswith(".sol")]
    for file_path in tqdm(repo_files, desc=f"Processing files in {repo['path']}", leave=False):
        statements, pragma_version, content = extract_statements(file_path)
        for statement_type, predicate in statements:
            message = ""
            if "," in predicate:
                predicate, message = predicate.rsplit(",", 1)
            
            results = analyze_context(content, statement_type, predicate)
            code_block, placement, contract_id, block_id =  results if results else ("", "", "", "")
            #print(f"Processed statement: type={statement_type}, predicate={predicate}, code_block={code_block}, contract_id={contract_id}, block_id={block_id}, file={file_path}")
            data.append({
                "protocol": repo["path"],
                "statement-type": statement_type,
                "predicate": predicate.strip(),
                "message": message.strip(),
                "code_block": code_block,
                "placement": placement,
                "pragma_version": pragma_version,
                "contract_id": contract_id,
                "block_id": block_id,
                "path": file_path
            })

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data)
output_dir = Path("datasets")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "require_assert_statements.csv"
df.to_csv(output_path, index=False)

print(f"Data extracted and saved to {output_path}")
print(df.head(10))


Processing repositories:   0%|          | 0/17 [00:00<?, ?it/s]

Processing repositories:   6%|▌         | 1/17 [00:00<00:02,  5.46it/s]

found a function!     ----->     function sendValue(address payable recipient, uint256 amount) internal {
found a library!     -----> library Address {
found a function!     ----->     function sendValue(address payable recipient, uint256 amount) internal {
found a library!     -----> library Address {
found a function!     ----->     function functionCallWithValue(address target, bytes memory data, uint256 value, string memory errorMessage) internal returns (bytes memory) {
found a library!     -----> library Address {
found a function!     ----->     function functionCallWithValue(address target, bytes memory data, uint256 value, string memory errorMessage) internal returns (bytes memory) {
found a library!     -----> library Address {
found a function!     ----->     function functionCallWithValue(address target, bytes memory data, uint256 value, string memory errorMessage) internal returns (bytes memory) {
found a library!     -----> library Address {
found a function!     ----->  

found a function!     ----->   function permit(
found a contract!     -----> contract MintableERC20 is IERC20WithPermit, ERC20 {
found a function!     ----->   function permit(
found a contract!     -----> contract MintableERC20 is IERC20WithPermit, ERC20 {
found a function!     ----->   function permit(
found a contract!     -----> contract MintableERC20 is IERC20WithPermit, ERC20 {
found a function!     ----->   function _setAssetsSources(address[] memory assets, address[] memory sources) internal {
found a contract!     -----> contract AaveOracle is IAaveOracle {
found a function!     ----->   function withdraw(uint256 wad) public {
found a contract!     -----> contract WETH9 {
found a function!     ----->   function transferFrom(address src, address dst, uint256 wad) public returns (bool) {
found a contract!     -----> contract WETH9 {
found a function!     ----->   function transferFrom(address src, address dst, uint256 wad) public returns (bool) {
found a contract!     -----> con

Processing repositories:  12%|█▏        | 2/17 [00:00<00:02,  6.01it/s]

...

found a function!     ----->     function pancakeV3SwapCallback(
found a contract!     -----> contract SwapRouter is
found a function!     ----->     function exactInputSingle(ExactInputSingleParams calldata params)
found a contract!     -----> contract SwapRouter is
found a function!     ----->     function exactInputSingle(ExactInputSingleParams calldata params)
found a contract!     -----> contract SwapRouter is
found a function!     ----->     function exactOutputInternal(
found a contract!     -----> contract SwapRouter is
found a function!     ----->     function exactOutputSingle(ExactOutputSingleParams calldata params)
found a contract!     -----> contract SwapRouter is
found a function!     ----->     function exactOutputSingle(ExactOutputSingleParams calldata params)
found a contract!     -----> contract SwapRouter is
found a function!     ----->     function positions(uint256 tokenId)
found a contract!     -----> contract NonfungiblePositionManager is
found a function!     -

found a function!     ----->     function handleRequest(address _to, bytes memory payload, bytes32 requestType) internal {
found a contract!     -----> contract ForwarderTest is BaseTest {
found a contract!     -----> contract ERC2771Helper is Test {
found a function!     ----->     function withdraw(uint256 wad) public {
found a contract!     -----> contract MockWETH is IWETH {
found a function!     ----->     function transferFrom(address src, address dst, uint256 wad) public returns (bool) {
found a contract!     -----> contract MockWETH is IWETH {
found a function!     ----->     function transferFrom(address src, address dst, uint256 wad) public returns (bool) {
found a contract!     -----> contract MockWETH is IWETH {
found a function!     ----->     function _addTokenTo(address _to, uint256 _tokenId) internal {
found a contract!     -----> contract VotingEscrow is IVotingEscrow, ERC2771Context, ReentrancyGuard {
found a function!     ----->     function _mint(address _to, uint25

found a contract!     -----> abstract contract VetoGovernor is Context, ERC165, EIP712, IVetoGovernor, IERC721Receiver, IERC1155Receiver {
found a function!     ----->     function propose(
found a contract!     -----> abstract contract VetoGovernor is Context, ERC165, EIP712, IVetoGovernor, IERC721Receiver, IERC1155Receiver {
found a function!     ----->     function propose(
found a contract!     -----> abstract contract VetoGovernor is Context, ERC165, EIP712, IVetoGovernor, IERC721Receiver, IERC1155Receiver {
found a function!     ----->     function propose(
found a contract!     -----> abstract contract VetoGovernor is Context, ERC165, EIP712, IVetoGovernor, IERC721Receiver, IERC1155Receiver {
found a function!     ----->     function propose(
found a contract!     -----> abstract contract VetoGovernor is Context, ERC165, EIP712, IVetoGovernor, IERC721Receiver, IERC1155Receiver {
found a function!     ----->     function cancel(
found a contract!     -----> abstract contract Veto

Processing repositories: 100%|██████████| 17/17 [00:07<00:00,  2.39it/s]


Data extracted and saved to datasets/require_assert_statements.csv
   protocol statement-type                                          predicate  \
0  lido-dao        require                    address(this).balance >= amount   
1  lido-dao        require            success, "Address: unable to send value   
2  lido-dao        require                     address(this).balance >= value   
3  lido-dao        require                                 isContract(target)   
4  lido-dao        require                                 isContract(target)   
5  lido-dao        require                                 isContract(target)   
6  lido-dao         assert  _IMPLEMENTATION_SLOT == bytes32(uint256(keccak...   
7  lido-dao        require              Address.isContract(newImplementation)   
8  lido-dao        require                                admin != address(0)   
9  lido-dao        require                                msg.sender == admin   

                                         

## Finding function details dataset        

In [ ]:
import os
import re
import subprocess
import pandas as pd
from pathlib import Path

# List of repositories to analyze
repositories = [
    {"url": "https://github.com/lidofinance/lido-dao", "path": "lido-dao"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/Layr-Labs/eigenlayer-contracts", "path": "eigenlayer-contracts"},
    {"url": "https://github.com/makerdao/dss", "path": "dai-stablecoin-system"},
    {"url": "https://github.com/rocket-pool/rocketpool", "path": "rocketpool"},
    {"url": "https://github.com/Uniswap/v3-core", "path": "uniswap-v3-core"},
    {"url": "https://github.com/Instadapp/dsa-resolvers", "path": "instadapp-dsa-resolvers"},
    {"url": "https://github.com/Instadapp/dsa-connectors", "path": "instadapp-dsa-connectors"},
    {"url": "https://github.com/compound-finance/compound-protocol", "path": "compound-protocol"},
    {"url": "https://github.com/aave/protocol-v2", "path": "aave-protocol-v2"},
    {"url": "https://github.com/aave/aave-v3-core", "path": "aave-v3-core"},
    {"url": "https://github.com/VenusProtocol/venus-protocol", "path": "venus-protocol"},
    {"url": "https://github.com/pancakeswap/pancake-smart-contracts", "path": "pancake-smart-contracts"},
    {"url": "https://github.com/pancakeswap/pancake-v4-core", "path": "pancake-v4-core"},
    {"url": "https://github.com/FraxFinance/frax-solidity", "path": "frax-solidity"},
    {"url": "https://github.com/pancakeswap/pancake-v3-contracts", "path": "pancake-v3-contracts"},
    {"url": "https://github.com/aerodrome-finance/contracts", "path": "aerodrome-finance-contracts"}
]

# Function to clone a repository
def clone_repository(repo_url, repo_path):
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", repo_url, repo_path])

# Function to remove comments from Solidity code
def remove_comments(content):
    content = re.sub(r'//.*', '', content)
    content = re.sub(r'/\*[\s\S]*?\*/', '', content)
    return content

# Function to extract function details from a file
def extract_function_details(file_path):
    with open(file_path, 'r') as file:
        content = file.read()

    # Remove comments
    content = remove_comments(content)

    # Regex to find contract, library, interface, and abstract contract identifiers
    contract_pattern = re.compile(r'\b(contract|library|interface|abstract\s+contract)\s+(\w+)')

    # Regex to find function definitions
    function_pattern = re.compile(r'function\s+(\w+)\s*\((.*?)\)\s*([^{]*)\{')

    # Find contract identifiers
    contract_identifiers = contract_pattern.findall(content)

    # Find functions
    functions = function_pattern.findall(content)

    # Process each function to extract details
    function_details = []
    for function_name, arguments, modifiers in functions:
        is_payable = 'payable' in modifiers
        is_override = 'override' in modifiers
        is_virtual = 'virtual' in modifiers

        visibility = 'public' if 'public' in modifiers else \
                     'external' if 'external' in modifiers else \
                     'internal' if 'internal' in modifiers else \
                     'private' if 'private' in modifiers else 'default'

        state_changing = 'pure' if 'pure' in modifiers else \
                         'view' if 'view' in modifiers else 'state-changing'

        # Extract return types
        returns_match = re.search(r'returns\s*\((.*?)\)', modifiers)
        returns = returns_match.group(1) if returns_match else ""

        # Remove keywords from modifiers and exclude anything after 'returns'
        modifiers = re.sub(r'returns\s*\(.*?\)', '', modifiers)
        actual_modifiers = [mod for mod in re.split(r'\W+', modifiers) 
                            if mod not in {'public', 'private', 'internal', 'external', 
                                           'view', 'pure', 'returns', 'override', 'payable', 'virtual'}]
        actual_modifiers = ', '.join(actual_modifiers)

        # Find the contract identifier that contains this function
        contract_id = ""
        for contract_type, contract_name in contract_identifiers:
            if contract_name in content:
                contract_id = contract_name
                break

        function_details.append({
            "function_name": function_name,
            "arguments": f'"{arguments}"',
            "payable": is_payable,
            "override": is_override,
            "virtual": is_virtual,
            "visibility": visibility,
            "state-changing": state_changing,
            "modifiers": actual_modifiers,
            "returns": returns,
            "path": file_path,
            "contract_id": contract_id
        })

    return function_details

# Traverse the directory and extract function details
data = []
protocols_dir = Path("protocols")
protocols_dir.mkdir(exist_ok=True)
for repo in repositories:
    repo_path = protocols_dir / repo["path"]
    clone_repository(repo["url"], str(repo_path))
    for root, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".sol"):
                file_path = os.path.join(root, file)
                function_details = extract_function_details(file_path)
                for detail in function_details:
                    detail["protocol"] = repo["path"]
                    data.append(detail)

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data)
output_dir = Path("datasets")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "function_details.csv"
df.to_csv(output_path, index=False)

print(f"Data extracted and saved to {output_path}")
print(df.head(10))